# TFG: RAG para Selección de Respuesta Terapéutica

MentalRiskES@IberLEF 2026, Tarea 2

Autora: Julia Acurio Ajila

Tutoras:  Natalia Madrueño y Soto Montalvo


**¿Cómo funciona el servidor?**

Atributos de cada mensaje:

- id: identificador interno de la sesión (cada sesión pertenece a un usuario distinto)
- round: número de ronda (del 1 hasta un número desconocido)
- patient_input: mensaje del paciente
- option_1, 2, 3: 3 posibles respuestas del terapeuta en ese contexto

Cosas importantes:

La primera ronda contiene todos los usuarios (porque todos tienen al menos un mensaje). Después de algunas rondas, algunos usuarios desaparecen del servidor (los que tienen menos mensajes).

El servidor no avisa de cuándo es el último mensaje de un usuario
Sabes que has terminado cuando el servidor devuelve una lista vacía
Después de cada petición, tu sistema tiene que devolver su predicción (option 1, 2 o 3) para cada usuario



## Enfoque
El sistema sigue el siguiente flujo en cada ronda:

1. Solicita al servidor el mensaje del paciente y las 3 opciones de respuesta
2. Recupera fragmentos relevantes de la base de conocimiento clínico (TCC, ACT, Entrevista Motivacional)
3. Selecciona la respuesta más adecuada mediante un LLM
4. Envía la predicción al servidor
5. Repite hasta que el servidor devuelve lista vacía

Estructura del notebook
1. Instalación de librerías
2. Configuración
3. Base de conocimiento clínico
4. Carga del trial corpus
5. Sistema RAG
6. Selección de respuesta con LLM
7. Bucle principal
8. Evaluación y análisis de resultados

In [1]:
# Instalación de librerías
!pip install chromadb pypdf sentence-transformers groq -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6